# Credit Risk Prediction

This notebook demonstrates a project for predicting loan default risk using a dataset of loan applicants. The key steps include data preprocessing, feature engineering, model training, and evaluating a Random Forest classifier.

---

## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score



---

## 2. Load and Inspect Data


In [ ]:
# Load the data
application_df = pd.read_csv("application_record.csv")
credit_df = pd.read_csv("credit_record.csv")

# View shape and sample
print("Application Data Shape:", application_df.shape)
print("Credit Data Shape:", credit_df.shape)
application_df.head()



---

## 3. Examine and Clean Columns


In [ ]:
# Check missing values
application_df.isnull().sum()

# Drop columns with too many missing values (e.g., 'OCCUPATION_TYPE')
application_df.drop(columns=["OCCUPATION_TYPE"], inplace=True)



---

## 4. Create Target Variable


In [ ]:
# Flag default: 1 if STATUS has 2, 3, 4, or 5; 0 otherwise
credit_df['default'] = credit_df['STATUS'].isin(['2', '3', '4', '5']).astype(int)

# Aggregate to one default status per ID
status_df = credit_df.groupby("ID")["default"].max().reset_index()



---

## 5. Merge Data


In [ ]:
merged_df = pd.merge(application_df, status_df, on="ID", how="inner")
merged_df.shape



---

## 6. Feature Engineering


In [ ]:
# One-hot encode categorical features
cat_cols = ["CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "NAME_EDUCATION_TYPE"]
merged_df = pd.get_dummies(merged_df, columns=cat_cols, drop_first=True)

# Drop ID column for modeling
model_df = merged_df.drop(columns=["ID"])



---

## 7. Train-Test Split


In [ ]:
X = model_df.drop("default", axis=1)
y = model_df["default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)



---

## 8. Model Training


In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)



---

## 9. Model Evaluation


In [ ]:
y_pred = rf_model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]))



---

## 10. Predict on Full Data


In [ ]:
default_probs = rf_model.predict_proba(X)[:, 1]
X["default_probability"] = default_probs
X.head()



---

## 11. Export Predictions


In [ ]:
# Create a DataFrame with IDs and predicted probabilities
time_index = merged_df.index
df_predictions = pd.DataFrame({
    "ID": merged_df.loc[time_index, "ID"],
    "default_probability": default_probs
})
# Save to CSV for reporting or downstream use
df_predictions.to_csv("default_probabilities.csv", index=False)
